# Week 4: Data Quality and Reproducible Pipelines

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-04-data-pipeline.ipynb)

**Estimated time:** 90–120 minutes  
**Deliverable:** a data-quality audit, a baseline, an improved pipeline, and one evidence-based conclusion.

### Learning goals

- Identify missing values and categorical columns.
- Fit preprocessing only on training data.
- Compare a simple baseline with a reproducible model pipeline.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 2. Create a synthetic renewal dataset

The rows represent fictional subscriptions. The dataset includes missing values and a categorical plan column so the notebook can demonstrate a complete preprocessing workflow.


In [ ]:
rng = np.random.default_rng(42)
n = 420

data = pd.DataFrame({
    "sessions_per_week": rng.poisson(4.5, n).astype(float),
    "support_tickets": rng.poisson(1.3, n).astype(float),
    "monthly_spend": rng.normal(42, 12, n).round(2),
    "plan": rng.choice(["basic", "plus", "pro"], n, p=[0.5, 0.35, 0.15]),
})

renewal_signal = (
    0.55 * data["sessions_per_week"]
    - 0.75 * data["support_tickets"]
    + 0.03 * data["monthly_spend"]
    + data["plan"].map({"basic": -0.4, "plus": 0.2, "pro": 0.7})
    + rng.normal(0, 1.2, n)
)
data["renewed"] = (renewal_signal > 2.6).astype(int)

for column in ["sessions_per_week", "monthly_spend", "plan"]:
    missing_rows = rng.choice(data.index, size=18, replace=False)
    data.loc[missing_rows, column] = np.nan

data.head()


## 3. Audit data quality


In [ ]:
audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "unique": data.nunique(dropna=True),
})

print(audit)
print("\nclass balance:")
print(data["renewed"].value_counts(normalize=True).sort_index())


## 4. Split before preprocessing

This order prevents information from the test set from influencing imputers, scalers, or encoders.


In [ ]:
X = data.drop(columns="renewed")
y = data["renewed"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("training rows:", len(X_train))
print("test rows:", len(X_test))


## 5. Establish a baseline


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"Baseline F1:       {baseline_f1:.3f}")


## 6. Define preprocessing by column type


In [ ]:
numeric_features = ["sessions_per_week", "support_tickets", "monthly_spend"]
categorical_features = ["plan"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])


## 7. Combine preprocessing and the model


In [ ]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

model_accuracy = accuracy_score(y_test, predictions)
model_f1 = f1_score(y_test, predictions)

print(f"Pipeline accuracy: {model_accuracy:.3f}")
print(f"Pipeline F1:       {model_f1:.3f}")


## 8. Compare results


In [ ]:
comparison = pd.DataFrame({
    "model": ["Most-frequent baseline", "Preprocessing pipeline"],
    "accuracy": [baseline_accuracy, model_accuracy],
    "f1": [baseline_f1, model_f1],
})
print(comparison)

comparison.set_index("model").plot(kind="bar", figsize=(8, 4), color=["#9AA9B5", "#147D88"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Baseline and pipeline comparison")
plt.xticks(rotation=0)
plt.show()


## 9. Inspect errors


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["Did not renew", "Renewed"],
    cmap="Blues",
)
plt.title("Pipeline confusion matrix")
plt.show()


## 10. Your turn

Choose one change:

- Add a derived feature such as sessions per support ticket.
- Replace median imputation with mean imputation.
- Compare logistic regression with a decision tree.

Record the new validation result and explain whether the change helped.


## 11. Validation


In [ ]:
transformed = model.named_steps["preprocess"].transform(X_test)

assert transformed.shape[0] == X_test.shape[0]
assert not np.isnan(transformed).any()
assert model_f1 > baseline_f1
print("Validation passed. The pipeline handles missing and categorical data and beats the baseline F1 score.")


## Submission checklist

- Include the data-quality audit.
- Report baseline and pipeline metrics.
- Explain why splitting comes before preprocessing.
- Document one attempted improvement and its result.
